# Optimization

## Import Necesary Libraries

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

## Load + Batch Data

In [3]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

## Model

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
model = NeuralNetwork()
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## Hyperparameters

In [6]:
learning_rate = 1e-3 # how much to update model parameters at each batch/epoch.
batch_size = 64 # number of data samples propagated through the network before updating model parameters.
epochs = 5 # number of times to iterate over the entire dataset.

## Optimizer

In [8]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

## Full Implementation

### Train Loop

In [14]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

### Test Loop

In [13]:
def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

### Run Loop

In [17]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 40
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.552547  [   64/60000]
loss: 0.667746  [ 6464/60000]
loss: 0.455091  [12864/60000]
loss: 0.688068  [19264/60000]
loss: 0.615988  [25664/60000]
loss: 0.589231  [32064/60000]
loss: 0.636042  [38464/60000]
loss: 0.685805  [44864/60000]
loss: 0.672928  [51264/60000]
loss: 0.621492  [57664/60000]
Test Error: 
 Accuracy: 78.7%, Avg loss: 0.612128 

Epoch 2
-------------------------------
loss: 0.539058  [   64/60000]
loss: 0.654792  [ 6464/60000]
loss: 0.444665  [12864/60000]
loss: 0.679280  [19264/60000]
loss: 0.609267  [25664/60000]
loss: 0.582283  [32064/60000]
loss: 0.623540  [38464/60000]
loss: 0.679864  [44864/60000]
loss: 0.666179  [51264/60000]
loss: 0.611618  [57664/60000]
Test Error: 
 Accuracy: 79.0%, Avg loss: 0.602748 

Epoch 3
-------------------------------
loss: 0.526678  [   64/60000]
loss: 0.642828  [ 6464/60000]
loss: 0.435103  [12864/60000]
loss: 0.670974  [19264/60000]
loss: 0.602945  [25664/60000]
loss: 0.575768  [32064/600

# Save & Load Model

## Import Necessary Libraries

In [16]:
import torch
import torchvision.models as models

## Save Model

In [18]:
model = models.vgg16(weights='IMAGENET1K_V1')
torch.save(model.state_dict(), 'model/model_weights.pth')

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/salman/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:43<00:00, 12.8MB/s] 


## Load Model

In [19]:
model = models.vgg16(weights=None)
model.load_state_dict(torch.load('model/model_weights.pth'))
model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1